In [2]:
import numpy as np
import pandas as pd
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, LSTM, RepeatVector, Dense
from sklearn.preprocessing import StandardScaler

# Load readings in time order
df = pd.read_csv("LP IV Updated Datasets/AutoEncoder/Energy Consumption/energy consumption.csv")
df["Timestamp"] = pd.to_datetime(df.Timestamp, dayfirst=True)
df = df.sort_values("Timestamp")
features = df.drop(columns=["Timestamp", "Anomaly_Label"]).columns

# Fill missing measurements using earlier values, or zero for initial gaps
df[features] = df[features].ffill().fillna(0)

# Use earlier readings for training and later readings for testing
split = int(len(df) * 0.8)
train = df.iloc[:split]
test = df.iloc[split:]

# Learn scaling from normal training readings
normal_train = train[train.Anomaly_Label == 0]
scaler = StandardScaler()
scaler.fit(normal_train[features])
train_values = scaler.transform(train[features])
test_values = scaler.transform(test[features])
train_labels = train.Anomaly_Label.to_numpy()
window_size = 24

# Build training windows containing only normal readings
x_train = []
for end in range(window_size, len(train_values) + 1):
    start = end - window_size
    if train_labels[start:end].sum() == 0:
        x_train.append(train_values[start:end])
x_train = np.array(x_train)

# Build all test windows
x_test = []
for end in range(window_size, len(test_values) + 1):
    start = end - window_size
    x_test.append(test_values[start:end])
x_test = np.array(x_test)

# Create LSTM Autoencoder
model = Sequential([
    Input(shape=(window_size, len(features))),
    LSTM(16),
    RepeatVector(window_size),
    LSTM(16, return_sequences=True),
    Dense(len(features))
])
model.compile(optimizer="adam", loss="mse")
model.fit(x_train, x_train, epochs=10, batch_size=128)

# Set a threshold using normal training errors
train_prediction = model.predict(x_train)
train_error = np.mean((x_train - train_prediction) ** 2, axis=(1, 2))
threshold = np.percentile(train_error, 95)

# Detect unusual test windows
test_prediction = model.predict(x_test)
test_error = np.mean((x_test - test_prediction) ** 2, axis=(1, 2))
anomalies = test_error > threshold
print("Anomalies:", anomalies.sum())


Epoch 1/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - loss: 0.9515
Epoch 2/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.9338
Epoch 3/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.9182
Epoch 4/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.9060
Epoch 5/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.8976
Epoch 6/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.8928
Epoch 7/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.8887
Epoch 8/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.8846
Epoch 9/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.8796
Epoch 10/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.8725
56/56 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
Anomalies: 111
